# Partitioning

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

In [ ]:
from pyspark.sql import types as T  # noqa: F401
import pyspark.sql.functions as F
from pyspark.sql import SparkSession

In [ ]:
spark = (
    SparkSession
    .builder
    .config("spark.driver.memory", "10g")
    .master("local[*]")
    .appName("6_0_partitioning")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("ERROR")

In [ ]:
listening_activity_file = "../data/partitioning/raw/Spotify_Listening_Activity.csv"
df_listening_actv = spark.read.csv(listening_activity_file, header=True, inferSchema=True)
df_listening_actv.show(5, False)

In [ ]:
df_listening_actv = (
    df_listening_actv
    .withColumnRenamed("listen_date", "listen_time")
    .withColumn("listen_date", F.to_date("listen_time", "yyyy-MM-dd HH:mm:ss.SSSSSS"))
    .withColumn("listen_hour", F.hour("listen_time"))
)

df_listening_actv.show(5, False)
df_listening_actv.printSchema()
df_listening_actv.count()

## Partitioning By `listen_date`

Let's say we want to **analyse the listening behaviours of user over time**. If we're given the complete dataset (with no partitions), Spark would scan the whole dataset for finding a particular date (similar to the bookshelf analogy where you would scan the entire bookself for finding a book if it is not organized). Given that our usecase needs analysis by date, partitioning (creating folders) on date would help Spark pin point to the exact folder. This makes searching very easy and Spark doesn't scan the entire dataset.  

In [ ]:
# Partitioning listening activity by the listen date
(
    df_listening_actv
    .write
    .partitionBy("listen_date")
    .mode("overwrite")
    .parquet("../data/partitioning/partitioned/listening_activity_pt")
)

## Partition Pruning

In [ ]:
df_listening_actv_pt_pruned = spark.read.parquet("../data/partitioning/partitioned/listening_activity_pt")
df_listening_actv_pt_pruned.filter("listen_date = '2019-01-01'").explain()

## What Problems Does Partitioning Solve? 
1. `Fast Search (Query Performance)`: Spark will only process the relevant partition instead of the entire dataset (example above). This greatly reduces I/O and query execution time. 
2. `Parallelism / Resource Utilization`: Each core processes 1 partition; More number of partitions, more is the parallelism; again this does not mean we forcefully increase the number of partitions. Each partition should be `128MB` in size. 


# Partitioning Examples
1. Single/multi level partitioning
2. Using `repartition`/`coalesce` with `partitionBy` (controlling number of files inside each partition): 
    - `parititionBy` affects how data is laid out in the storage and is going to ensure that the output directory is organized into subdirectories based on the `value` given in `partitionBy`.  
    - Number of files in each `value` directory of `partitionBy` depends on the number supplied in the `repartition`/`coalesce`.

#### 1. Single/multi level partitioning

In [ ]:
(
    df_listening_actv
    .write
    .mode("overwrite")
    .partitionBy("listen_date", "listen_hour")
    .parquet("../data/partitioning/partitioned/listening_activity_pt_2")
)

In [ ]:
(
    df_listening_actv
    .write
    .mode("overwrite")
    .partitionBy("listen_hour", "listen_date")
    .parquet("../data/partitioning/partitioned/listening_activity_pt_3")
)

#### 2. Using `repartition`/`coalesce` with `partitionBy`

In [ ]:
(
    df_listening_actv
    .repartition(3)
    .write
    .mode("overwrite")
    .partitionBy("listen_date")
    .parquet("../data/partitioning/partitioned/listening_activity_pt_4")
)

In [ ]:
# The coalesce method reduces the number of partitions in a DataFrame.
# It avoids full shuffle, instead of creating new partitions, it shuffles the data using default Hash Partitioner,
# and adjusts into existing partitions, this means it can only decrease the number of partitions.

(
    df_listening_actv
    .coalesce(3)
    .write
    .mode("overwrite")
    .partitionBy("listen_date")
    .parquet("../data/partitioning/partitioned/listening_activity_pt_5")
)

## Experimenting With `spark.sql.files.maxPartitionBytes`

In [ ]:
spark.stop()
spark = SparkSession.builder.appName("Test spark.sql.files.maxPartitionBytes").getOrCreate()

df_default = spark.read.csv("../data/partitioning/raw/listening_activity.csv", header=True, inferSchema=True)
default_partitions = df_default.rdd.getNumPartitions()
print(f"Number of partitions with default maxPartitionBytes: {default_partitions}")


In [ ]:
spark.conf.set("spark.sql.files.maxPartitionBytes", "1000")

df_modified = spark.read.csv("../data/partitioning/raw/listening_activity.csv", header=True, inferSchema=True)
modified_partitions = df_modified.rdd.getNumPartitions()
print(f"Number of partitions with modified maxPartitionBytes: {modified_partitions}")

In [ ]:
spark.stop()